Generating the dependencies for the EDA

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ── FIX: Added all missing imports ──────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans

import geohash2

# Update these paths to match your local setup
data = pd.read_csv(r'C:\Users\Venkat\Desktop\IIITB\flip hackathon\Traffic-demand-prediction\dataset\train.csv')
test_data = pd.read_csv(r'C:\Users\Venkat\Desktop\IIITB\flip hackathon\Traffic-demand-prediction\dataset\test.csv')

print('Data loaded successfully!')
print(f'Train shape: {data.shape}')
print(f'Test shape:  {test_data.shape}')

In [ ]:
print(data.head()) #to have an idea about how the columns actually look like
print('\nColumn dtypes:')
data.info()

Here we can see that we have some missing data, let us see how much of our data is actually missing

In [ ]:
print('Missing values in training data:')
missing_data = data.isnull().sum()
print(missing_data[missing_data > 0])

print('\nMissing values in testing data:')
missing_test = test_data.isnull().sum()
print(missing_test[missing_test > 0])

So now, we just have to fill in the missing values(mode for weather and road whereas median for temperature)

In [ ]:
# Compute fill values from train only (no leakage)
weather_mode = data['Weather'].mode()[0]
road_mode    = data['RoadType'].mode()[0]
temp_median  = data['Temperature'].median()

print(f'Weather mode:       {weather_mode}')
print(f'RoadType mode:      {road_mode}')
print(f'Temperature median: {temp_median:.4f}')

for df in [data, test_data]:
    df['Weather']     = df['Weather'].fillna(weather_mode)
    df['RoadType']    = df['RoadType'].fillna(road_mode)
    df['Temperature'] = df['Temperature'].fillna(temp_median)

print('\nMissing values filled successfully!')

In [ ]:
def parse_timestamp(df):
    """Extract hour/minute and cyclical encodings from 'H:M' timestamp."""
    parts = df['timestamp'].str.split(':', expand=True).astype(int)
    df = df.copy()
    df['hour']        = parts[0]
    df['minute']      = parts[1]
    # Cyclical time features
    df['hour_sin']    = np.sin(2 * np.pi * df['hour']   / 24)
    df['hour_cos']    = np.cos(2 * np.pi * df['hour']   / 24)
    df['minute_sin']  = np.sin(2 * np.pi * df['minute'] / 60)   # NEW
    df['minute_cos']  = np.cos(2 * np.pi * df['minute'] / 60)   # NEW
    df['day_sin']     = np.sin(2 * np.pi * df['day']    / 7)
    df['day_cos']     = np.cos(2 * np.pi * df['day']    / 7)
    # NEW: Derived time features
    df['is_peak_hour']  = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
    df['is_night']      = df['hour'].isin(range(0, 6)).astype(int)
    df['hour_x_day']    = df['hour'] * df['day']               # interaction
    return df

def decode_geohash(df):
    df = df.copy()
    decoded        = df['geohash'].astype(str).apply(lambda gh: geohash2.decode(gh))
    df['lat']      = decoded.apply(lambda x: float(x[0])).astype(np.float64)
    df['lon']      = decoded.apply(lambda x: float(x[1])).astype(np.float64)
    return df

data      = parse_timestamp(data)
test_data = parse_timestamp(test_data)

data      = decode_geohash(data)
test_data = decode_geohash(test_data)

print('Timestamp & geohash features done!')
print(data[['hour','minute','hour_sin','hour_cos','day_sin','day_cos','lat','lon']].head())

In [ ]:
# ── FIX: Create geo_cluster_4 and geo_cluster_5 — were referenced but never built ──
coords_train = data[['lat', 'lon']].values
coords_test  = test_data[['lat', 'lon']].values

for n in [4, 5]:
    col = f'geo_cluster_{n}'
    km  = KMeans(n_clusters=n, random_state=42, n_init=10)
    km.fit(coords_train)
    data[col]      = km.predict(coords_train)
    test_data[col] = km.predict(coords_test)

print('Geo-clusters created: geo_cluster_4, geo_cluster_5')
print(data[['lat','lon','geo_cluster_4','geo_cluster_5']].head())

In [ ]:
# ── FIX: Ordinal-encode all string columns ───────────────────────────────────
from sklearn.preprocessing import OrdinalEncoder

cat_cols = ['Weather', 'RoadType', 'LargeVehicles', 'Landmarks']
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

combined = pd.concat([data[cat_cols], test_data[cat_cols]])
enc.fit(combined)

data[cat_cols]      = enc.transform(data[cat_cols]).astype(int)
test_data[cat_cols] = enc.transform(test_data[cat_cols]).astype(int)

print('Categorical encoding done!')
for col, cats in zip(cat_cols, enc.categories_):
    print(f'  {col}: {list(cats)}')

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# NEW: added std-based features for demand volatility
encoding_configs = [
    ('geo_hour_mean_demand', ['geohash', 'hour'],  'mean'),
    ('geo_day_mean_demand',  ['geohash', 'day'],   'mean'),
    ('geo_mean_demand',      ['geohash'],           'mean'),
    ('geo_hour_std_demand',  ['geohash', 'hour'],  'std'),   # NEW
    ('geo_day_std_demand',   ['geohash', 'day'],   'std'),   # NEW
]

for col_group, merge_keys, agg_fn in encoding_configs:
    data[col_group] = np.nan
    for train_idx, val_idx in kf.split(data):
        fold_stat = (
            data.iloc[train_idx]
            .groupby(merge_keys)['demand']
            .agg(agg_fn)
            .reset_index()
            .rename(columns={'demand': col_group})
        )
        data.iloc[val_idx, data.columns.get_loc(col_group)] = (
            data.iloc[val_idx][merge_keys]
            .merge(fold_stat, on=merge_keys, how='left')[col_group]
            .values
        )

# Build full-train lookup tables for test set
for col_group, merge_keys, agg_fn in encoding_configs:
    lookup = (
        data.groupby(merge_keys)['demand']
        .agg(agg_fn)
        .reset_index()
        .rename(columns={'demand': col_group})
    )
    test_data = test_data.merge(lookup, on=merge_keys, how='left')

# Fill NaN for unseen combos in test
# ── FIX: replace deprecated inplace=True with direct assignment ─────────────
for col_group, _, _ in encoding_configs:
    fill_val = data[col_group].mean()
    data[col_group]      = data[col_group].fillna(fill_val)
    test_data[col_group] = test_data[col_group].fillna(fill_val)

print('K-Fold target encoding (mean + std) done!')

In [ ]:
# ── FIX: LabelEncoder import was missing — now added at the top ──────────────
gh_le       = LabelEncoder()
combined_gh = pd.concat([data['geohash'], test_data['geohash']]).astype(str)
gh_le.fit(combined_gh)
data['geohash']      = gh_le.transform(data['geohash'].astype(str))
test_data['geohash'] = gh_le.transform(test_data['geohash'].astype(str))

feature_cols = [
    'geohash',
    'day', 'day_sin', 'day_cos',
    'hour', 'minute', 'hour_sin', 'hour_cos',
    'minute_sin', 'minute_cos',              # NEW
    'is_peak_hour', 'is_night',              # NEW
    'hour_x_day',                            # NEW
    'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks',
    'Temperature', 'Weather',
    'lat', 'lon',
    'geo_cluster_4', 'geo_cluster_5',        # FIX: now actually created
    'geo_hour_mean_demand', 'geo_day_mean_demand', 'geo_mean_demand',
    'geo_hour_std_demand', 'geo_day_std_demand',   # NEW
]

X      = data[feature_cols]
y      = data['demand']
X_test = test_data[feature_cols]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train set : {X_train.shape}')
print(f'Val set   : {X_val.shape}')
print(f'Test set  : {X_test.shape}')
print('\nFeature dtypes:')
print(X_train.dtypes)

In [ ]:
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# ── ACCURACY BOOST: increased n_estimators, tuned leaves & regularisation ───
lgb_model = LGBMRegressor(
    n_estimators     = 2000,      # was 500
    learning_rate    = 0.02,      # slightly lower for more trees
    num_leaves       = 127,       # was 63 — captures more structure
    min_child_samples= 15,
    max_depth        = -1,
    subsample        = 0.85,
    colsample_bytree = 0.85,
    reg_alpha        = 0.05,
    reg_lambda       = 0.2,
    random_state     = 42,
    n_jobs           = -1,
    verbose          = -1
)

xgb_model = XGBRegressor(
    n_estimators     = 2000,      # was 500
    learning_rate    = 0.02,
    max_depth        = 7,         # was 6
    min_child_weight = 3,
    subsample        = 0.85,
    colsample_bytree = 0.85,
    reg_alpha        = 0.05,
    reg_lambda       = 0.2,
    random_state     = 42,
    n_jobs           = -1,
    eval_metric      = 'rmse',
    early_stopping_rounds = 100
)

cat_model = CatBoostRegressor(
    iterations       = 2000,      # was 500
    learning_rate    = 0.02,
    depth            = 8,         # was 6
    l2_leaf_reg      = 3,
    min_data_in_leaf = 15,
    random_seed      = 42,
    verbose          = 200
)

print('Training LightGBM...')
lgb_model.fit(
    X_train, y_train,
    eval_set  = [(X_val, y_val)],
    callbacks = [early_stopping(100), log_evaluation(200)]
)
lgb_val_preds = lgb_model.predict(X_val)
lgb_r2 = r2_score(y_val, lgb_val_preds)
print(f'LightGBM Validation R²: {lgb_r2 * 100:.2f}%')

print('\nTraining XGBoost...')
xgb_model.fit(
    X_train, y_train,
    eval_set = [(X_val, y_val)],
    verbose  = False
)
xgb_val_preds = xgb_model.predict(X_val)
xgb_r2 = r2_score(y_val, xgb_val_preds)
print(f'XGBoost  Validation R²: {xgb_r2 * 100:.2f}%')

print('\nTraining CatBoost...')
cat_model.fit(
    X_train, y_train,
    eval_set              = (X_val, y_val),
    early_stopping_rounds = 100
)
cat_val_preds = cat_model.predict(X_val)
cat_r2 = r2_score(y_val, cat_val_preds)
print(f'CatBoost Validation R²: {cat_r2 * 100:.2f}%')

In [ ]:
from scipy.optimize import minimize

# ── ACCURACY BOOST: optimise blend weights instead of using fixed 0.4/0.3/0.3 ──
val_matrix = np.column_stack([lgb_val_preds, xgb_val_preds, cat_val_preds])

def neg_r2(weights):
    w = np.array(weights)
    w = np.clip(w, 0, None)
    w /= w.sum()
    return -r2_score(y_val, val_matrix @ w)

result = minimize(
    neg_r2,
    x0     = [0.4, 0.3, 0.3],
    method = 'Nelder-Mead',
    options= {'maxiter': 1000, 'xatol': 1e-6}
)

opt_w = np.clip(result.x, 0, None)
opt_w /= opt_w.sum()
print(f'Optimised weights — LGB: {opt_w[0]:.3f}  XGB: {opt_w[1]:.3f}  CAT: {opt_w[2]:.3f}')

ensemble_val_preds = val_matrix @ opt_w
ensemble_r2 = r2_score(y_val, ensemble_val_preds)
print(f'\nEnsemble Validation R²: {ensemble_r2 * 100:.2f}%')

# Final test predictions
test_matrix      = np.column_stack([
    lgb_model.predict(X_test),
    xgb_model.predict(X_test),
    cat_model.predict(X_test)
])
final_test_preds = np.clip(test_matrix @ opt_w, 0, None)
print(f'Test preds  min: {final_test_preds.min():.4f}  max: {final_test_preds.max():.4f}')

In [ ]:
submission = pd.DataFrame({
    'Index':  test_data['Index'] if 'Index' in test_data.columns else range(len(final_test_preds)),
    'demand': final_test_preds
})

save_path = os.path.join(os.getcwd(), 'submission.csv')
submission.to_csv(save_path, index=False)

print(f'Submission shape: {submission.shape}')
print(submission.head(10))
print(f'\nsubmission.csv saved to: {os.path.abspath(save_path)}')